# CS-781 Phase 1 — Master End-to-End Notebook v8

This notebook is the **clean, resumable Architecture-A pipeline**.

It intentionally separates:
1. official data acquisition and audit;
2. leakage-safe development holdout;
3. legal tokenizer research;
4. large from-scratch Architecture A;
5. robust full-corpus token caching;
6. resumable multi-session training;
7. best-checkpoint selection;
8. official 1000-example inference;
9. `prediction.json` + `submission.zip`.

**Hard rule:** the model uses only the official Phase-1 corpus and is randomly initialized. The reserved 1,000-document holdout is never used for training.

The capstone still requires four substantially different architectures; this notebook establishes the fully tested A→submission pipeline before B/C/D are added.


In [ ]:

import os
import sys
import gc
import json
import math
import time
import random
import shutil
import zipfile
import importlib.util
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch

# Total Run-All safety budget; deliberately below the platform hard ceiling.
NOTEBOOK_START_TIME = time.time()
SAFE_TOTAL_RUNTIME_MINUTES = 650
import torch.nn as nn
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the intended training run.")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"VRAM={props.total_memory / 1024**3:.2f} GB"
    )


In [ ]:

SEED = 781

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if importlib.util.find_spec("ijson") is None:
    import subprocess
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ijson",
    ])

import ijson

print("Seed:", SEED)
print("ijson:", ijson.__version__ if hasattr(ijson, "__version__") else "installed")


In [ ]:

BASE_DIR = Path("/kaggle/working/CS781_PHASE1_RESEARCH")

DATA_DIR = BASE_DIR / "data"
TOKENIZER_DIR = BASE_DIR / "tokenizers"
RESULTS_DIR = BASE_DIR / "results"

TRAINING_DIR = BASE_DIR / "training"
TOKEN_CACHE_DIR = TRAINING_DIR / "token_cache_40k"
CHECKPOINT_DIR = TRAINING_DIR / "checkpoints_archA_large"
TRAIN_RESULTS_DIR = TRAINING_DIR / "results"

SUBMISSION_DIR = BASE_DIR / "submissions"

for d in [
    BASE_DIR,
    DATA_DIR,
    TOKENIZER_DIR,
    RESULTS_DIR,
    TRAINING_DIR,
    TOKEN_CACHE_DIR,
    CHECKPOINT_DIR,
    TRAIN_RESULTS_DIR,
    SUBMISSION_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("TOKEN_CACHE_DIR:", TOKEN_CACHE_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("SUBMISSION_DIR:", SUBMISSION_DIR)

print()
print("IMPORTANT: before long training, enable Kaggle notebook file persistence.")


In [ ]:

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF")
except Exception as e:
    raise RuntimeError(
        "Kaggle Secret 'HF' is required."
    ) from e

if not HF_TOKEN:
    raise RuntimeError(
        "Kaggle Secret 'HF' is empty."
    )

login(token=HF_TOKEN)

print("Hugging Face authentication: PASSED")


In [ ]:

PHASE1_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

TEST_FILENAME = "Phase_1_Pretraining/Test/test.json"

SOURCE_ORDER = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

downloaded_files = {}

for source, filename in PHASE1_FILES.items():

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    downloaded_files[source] = path

    print(
        f"{source:20s} -> {path}"
    )

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILENAME,
    repo_type="dataset",
    token=HF_TOKEN,
)

print("\nOfficial test:", test_path)


In [ ]:

profile_path = RESULTS_DIR / "data_profile.csv"

if profile_path.exists():

    profile_df = pd.read_csv(
        profile_path
    )

    print(
        "Loaded existing audit:",
        profile_path
    )

else:

    profiles = []

    PROFILE_SAMPLE_LIMIT = 10000

    for source_name in SOURCE_ORDER:

        path = downloaded_files[
            source_name
        ]

        print(
            "\nAnalyzing:",
            source_name
        )

        total_records = 0
        valid_records = 0
        invalid_records = 0

        total_characters = 0
        total_words = 0

        min_words = None
        max_words = 0

        min_chars = None
        max_chars = 0

        sampled_word_lengths = []

        rng = random.Random(
            SEED
            + sum(
                ord(c)
                for c in source_name
            )
        )

        started = time.time()

        with open(
            path,
            "rb"
        ) as f:

            for record in ijson.items(
                f,
                "item"
            ):

                total_records += 1

                if not isinstance(
                    record,
                    dict
                ):
                    invalid_records += 1
                    continue

                text = record.get(
                    "text",
                    ""
                )

                if (
                    not isinstance(
                        text,
                        str
                    )
                    or not text.strip()
                ):
                    invalid_records += 1
                    continue

                valid_records += 1

                char_count = len(text)

                provided_wc = record.get(
                    "word_count"
                )

                if isinstance(
                    provided_wc,
                    (int, float)
                ):
                    word_count = int(
                        provided_wc
                    )
                else:
                    word_count = len(
                        text.split()
                    )

                total_characters += char_count
                total_words += word_count

                min_words = (
                    word_count
                    if min_words is None
                    else min(
                        min_words,
                        word_count
                    )
                )

                max_words = max(
                    max_words,
                    word_count
                )

                min_chars = (
                    char_count
                    if min_chars is None
                    else min(
                        min_chars,
                        char_count
                    )
                )

                max_chars = max(
                    max_chars,
                    char_count
                )

                # Correct reservoir sampling.
                if len(
                    sampled_word_lengths
                ) < PROFILE_SAMPLE_LIMIT:

                    sampled_word_lengths.append(
                        word_count
                    )

                else:

                    j = rng.randrange(
                        valid_records
                    )

                    if (
                        j
                        < PROFILE_SAMPLE_LIMIT
                    ):
                        sampled_word_lengths[
                            j
                        ] = word_count

        arr = np.asarray(
            sampled_word_lengths,
            dtype=np.float64
        )

        profiles.append({
            "source": source_name,
            "total_records": total_records,
            "valid_records": valid_records,
            "invalid_records": invalid_records,
            "total_characters": total_characters,
            "total_words": total_words,
            "mean_words": (
                total_words / valid_records
                if valid_records
                else 0.0
            ),
            "min_words": (
                min_words or 0
            ),
            "max_words": max_words,
            "p50_words_approx": (
                float(np.percentile(arr, 50))
                if len(arr)
                else 0.0
            ),
            "p95_words_approx": (
                float(np.percentile(arr, 95))
                if len(arr)
                else 0.0
            ),
            "p99_words_approx": (
                float(np.percentile(arr, 99))
                if len(arr)
                else 0.0
            ),
            "mean_characters": (
                total_characters / valid_records
                if valid_records
                else 0.0
            ),
            "max_characters": max_chars,
            "audit_seconds": time.time() - started,
        })

    profile_df = pd.DataFrame(
        profiles
    )

    profile_df["word_share_pct"] = (
        100.0
        * profile_df["total_words"]
        / profile_df["total_words"].sum()
    )

    profile_df.to_csv(
        profile_path,
        index=False
    )

display(profile_df)
print("Saved:", profile_path)


In [ ]:

with open(
    test_path,
    "r",
    encoding="utf-8"
) as f:

    raw_test = json.load(f)

if isinstance(
    raw_test,
    dict
):

    test_records = [
        {
            "id": str(k),
            "text": (
                v.get("text", "")
                if isinstance(v, dict)
                else str(v)
            ),
        }
        for k, v in raw_test.items()
    ]

elif isinstance(
    raw_test,
    list
):

    test_records = []

    for i, record in enumerate(
        raw_test
    ):

        if isinstance(
            record,
            dict
        ):

            example_id = record.get(
                "id",
                record.get(
                    "example_id",
                    record.get(
                        "document_id",
                        i
                    )
                )
            )

            text = record.get(
                "text",
                ""
            )

        else:

            example_id = i
            text = str(record)

        test_records.append({
            "id": str(example_id),
            "text": text,
        })

else:

    raise TypeError(
        "Unsupported official test JSON type."
    )

test_word_counts = [
    len(r["text"].split())
    for r in test_records
]

print(
    "Official test examples:",
    len(test_records)
)

print(
    "Exactly 512 words:",
    sum(
        x == 512
        for x in test_word_counts
    )
)

assert len(test_records) == 1000
assert all(
    x == 512
    for x in test_word_counts
)

print(
    "Official test format: PASSED"
)


In [ ]:

from transformers import GPT2TokenizerFast

audit_tokenizer = GPT2TokenizerFast.from_pretrained(
    "gpt2"
)

# Only used as a tokenizer baseline.
audit_tokenizer.model_max_length = 10**9

gpt2_test_lengths = [
    len(
        audit_tokenizer.encode(
            r["text"],
            add_special_tokens=False
        )
    )
    for r in test_records
]

print(
    "GPT-2 vocab:",
    len(audit_tokenizer)
)

print(
    "Test token min/mean/max:",
    min(gpt2_test_lengths),
    float(np.mean(gpt2_test_lengths)),
    max(gpt2_test_lengths)
)


In [ ]:

HOLDOUT_PER_SOURCE = 250
MIN_HOLDOUT_WORDS = 640

holdout_manifest_path = (
    RESULTS_DIR
    / "phase1_holdout_manifest.json"
)

holdout_keys_path = (
    RESULTS_DIR
    / "phase1_holdout_keys.json"
)

if (
    holdout_manifest_path.exists()
    and holdout_keys_path.exists()
):

    with open(
        holdout_manifest_path,
        "r",
        encoding="utf-8"
    ) as f:

        holdout_manifest = json.load(f)

    with open(
        holdout_keys_path,
        "r",
        encoding="utf-8"
    ) as f:

        holdout_key_records = json.load(f)

    holdout_keys = {
        (
            x["source"],
            int(x["index"])
        )
        for x in holdout_key_records
    }

    print(
        "Reusing existing holdout."
    )

else:

    holdout_manifest = {}
    holdout_keys = set()

    for source_name in SOURCE_ORDER:

        rng = random.Random(
            SEED
            + 5000
            + sum(
                ord(c)
                for c in source_name
            )
        )

        reservoir = []
        eligible_count = 0

        with open(
            downloaded_files[source_name],
            "rb"
        ) as f:

            for index, record in enumerate(
                ijson.items(
                    f,
                    "item"
                )
            ):

                if not isinstance(
                    record,
                    dict
                ):
                    continue

                text = record.get(
                    "text",
                    ""
                )

                if (
                    not isinstance(
                        text,
                        str
                    )
                    or not text.strip()
                ):
                    continue

                words = text.split()

                if len(words) < (
                    MIN_HOLDOUT_WORDS
                ):
                    continue

                eligible_count += 1

                candidate = {
                    "source": source_name,
                    "index": int(index),
                    "title": record.get(
                        "title",
                        ""
                    ),
                    "date": record.get(
                        "date",
                        ""
                    ),
                    "court": record.get(
                        "court",
                        ""
                    ),
                }

                if len(
                    reservoir
                ) < HOLDOUT_PER_SOURCE:

                    reservoir.append(
                        candidate
                    )

                else:

                    j = rng.randrange(
                        eligible_count
                    )

                    if j < HOLDOUT_PER_SOURCE:

                        reservoir[j] = (
                            candidate
                        )

        holdout_manifest[
            source_name
        ] = reservoir

        for item in reservoir:

            holdout_keys.add(
                (
                    source_name,
                    int(item["index"])
                )
            )

        print(
            f"{source_name:20s} | "
            f"eligible={eligible_count:,} | "
            f"selected={len(reservoir)}"
        )

    assert len(
        holdout_keys
    ) == 1000

    with open(
        holdout_manifest_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            holdout_manifest,
            f,
            ensure_ascii=False,
            indent=2
        )

    with open(
        holdout_keys_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            [
                {
                    "source": s,
                    "index": int(i)
                }
                for s, i in sorted(
                    holdout_keys
                )
            ],
            f,
            indent=2
        )

print(
    "Holdout records:",
    len(holdout_keys)
)

assert len(
    holdout_keys
) == 1000


In [ ]:

dev_path = (
    RESULTS_DIR
    / "phase1_dev_examples.json"
)

if dev_path.exists():

    with open(
        dev_path,
        "r",
        encoding="utf-8"
    ) as f:

        dev_examples = json.load(f)

    print(
        "Loaded existing dev set:",
        dev_path
    )

else:

    dev_examples = []

    for source_name in SOURCE_ORDER:

        selected_indices = {
            int(x["index"])
            for x in holdout_manifest[
                source_name
            ]
        }

        found = 0

        with open(
            downloaded_files[source_name],
            "rb"
        ) as f:

            for index, record in enumerate(
                ijson.items(
                    f,
                    "item"
                )
            ):

                if index not in selected_indices:
                    continue

                text = record.get(
                    "text",
                    ""
                )

                words = text.split()

                prompt = " ".join(
                    words[:512]
                )

                reference = " ".join(
                    words[512:640]
                )

                assert (
                    len(prompt.split())
                    == 512
                )

                assert (
                    len(reference.split())
                    == 128
                )

                dev_examples.append({
                    "id": (
                        f"{source_name}__{index}"
                    ),
                    "source": source_name,
                    "index": int(index),
                    "prompt": prompt,
                    "reference": reference,
                })

                found += 1

                if found == HOLDOUT_PER_SOURCE:
                    break

    assert len(
        dev_examples
    ) == 1000

    with open(
        dev_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dev_examples,
            f,
            ensure_ascii=False,
            indent=2
        )

print(
    "Dev examples:",
    len(dev_examples)
)

assert len(
    dev_examples
) == 1000

assert all(
    len(x["prompt"].split()) == 512
    for x in dev_examples
)

assert all(
    len(x["reference"].split()) == 128
    for x in dev_examples
)

print(
    "512→128 dev benchmark: PASSED"
)


In [ ]:

TOKENIZER_SAMPLE_PER_SOURCE = 3000
TOKENIZER_SAMPLE_CHARS = 5000

TOKENIZER_SAMPLE_DIR = (
    DATA_DIR
    / "tokenizer_sample"
)

TOKENIZER_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sample_files = []

for source_name in SOURCE_ORDER:

    sample_path = (
        TOKENIZER_SAMPLE_DIR
        / f"{source_name}.txt"
    )

    if not sample_path.exists():

        rng = random.Random(
            SEED
            + 1000
            + sum(
                ord(c)
                for c in source_name
            )
        )

        reservoir = []
        eligible_count = 0

        with open(
            downloaded_files[source_name],
            "rb"
        ) as f:

            for index, record in enumerate(
                ijson.items(
                    f,
                    "item"
                )
            ):

                if (
                    source_name,
                    index
                ) in holdout_keys:
                    continue

                if not isinstance(
                    record,
                    dict
                ):
                    continue

                text = record.get(
                    "text",
                    ""
                )

                if (
                    not isinstance(
                        text,
                        str
                    )
                    or not text.strip()
                ):
                    continue

                eligible_count += 1

                candidate = (
                    text[
                        :TOKENIZER_SAMPLE_CHARS
                    ]
                    .replace(
                        "\x00",
                        " "
                    )
                )

                if len(
                    reservoir
                ) < TOKENIZER_SAMPLE_PER_SOURCE:

                    reservoir.append(
                        candidate
                    )

                else:

                    j = rng.randrange(
                        eligible_count
                    )

                    if (
                        j
                        < TOKENIZER_SAMPLE_PER_SOURCE
                    ):
                        reservoir[
                            j
                        ] = candidate

        with open(
            sample_path,
            "w",
            encoding="utf-8"
        ) as f:

            for text in reservoir:

                f.write(text)
                f.write("\n")

        print(
            f"Built {source_name}: "
            f"{len(reservoir):,} docs"
        )

    else:

        print(
            "Reusing:",
            sample_path
        )

    sample_files.append(
        str(sample_path)
    )

assert len(
    sample_files
) == 4


In [ ]:

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import (
    ByteLevel as ByteLevelPreTokenizer
)
from tokenizers.decoders import (
    ByteLevel as ByteLevelDecoder
)
from transformers import PreTrainedTokenizerFast


def train_or_load_legal_bpe(
    vocab_size,
    output_dir,
    files,
):

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    tokenizer_json = (
        output_dir
        / "tokenizer.json"
    )

    if tokenizer_json.exists():

        existing = (
            PreTrainedTokenizerFast
            .from_pretrained(
                output_dir
            )
        )

        # Validate an existing tokenizer so an old broken tokenizer
        # cannot silently survive a later Run All.
        try:
            probe_text = Path(files[0]).read_text(
                encoding="utf-8"
            )[:5000]

            probe_ids = existing.encode(
                probe_text,
                add_special_tokens=False
            )

            probe_decoded = existing.decode(
                probe_ids,
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False,
            )

            if probe_decoded == probe_text:
                return existing

            print(
                "Existing tokenizer failed round-trip; retraining:",
                output_dir
            )

        except Exception as exc:
            print(
                "Existing tokenizer validation failed; retraining:",
                output_dir,
                "|",
                exc,
            )

    tokenizer = Tokenizer(
        BPE(
            unk_token="<unk>"
        )
    )

    tokenizer.pre_tokenizer = (
        ByteLevelPreTokenizer(
            add_prefix_space=False
        )
    )

    tokenizer.decoder = (
        ByteLevelDecoder()
    )

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=[
            "<pad>",
            "<unk>",
            "<s>",
            "</s>",
        ],
    )

    tokenizer.train(
        files=files,
        trainer=trainer,
    )

    tokenizer.save(
        str(tokenizer_json)
    )

    fast = PreTrainedTokenizerFast(
        tokenizer_file=str(
            tokenizer_json
        ),
        unk_token="<unk>",
        pad_token="<pad>",
        bos_token="<s>",
        eos_token="</s>",
    )

    fast.save_pretrained(
        output_dir
    )

    return fast


print(
    "Legal BPE trainer: READY"
)


In [ ]:

LEGAL_32K_DIR = (
    TOKENIZER_DIR
    / "legal_bpe_32k"
)

LEGAL_40K_DIR = (
    TOKENIZER_DIR
    / "legal_bpe_40k"
)

LEGAL_50K_DIR = (
    TOKENIZER_DIR
    / "legal_bpe_50k"
)

legal_tokenizer_32k = (
    train_or_load_legal_bpe(
        32000,
        LEGAL_32K_DIR,
        sample_files
    )
)

legal_tokenizer_40k = (
    train_or_load_legal_bpe(
        40000,
        LEGAL_40K_DIR,
        sample_files
    )
)

legal_tokenizer_50k = (
    train_or_load_legal_bpe(
        50257,
        LEGAL_50K_DIR,
        sample_files
    )
)

assert len(
    legal_tokenizer_32k
) == 32000

assert len(
    legal_tokenizer_40k
) == 40000

assert len(
    legal_tokenizer_50k
) == 50257

print(
    "32K:",
    len(legal_tokenizer_32k)
)

print(
    "40K:",
    len(legal_tokenizer_40k)
)

print(
    "50K:",
    len(legal_tokenizer_50k)
)


In [ ]:

legal_tokenizers = {
    "LEGAL_32K": legal_tokenizer_32k,
    "LEGAL_40K": legal_tokenizer_40k,
    "LEGAL_50K": legal_tokenizer_50k,
}

balanced_examples = []

for source_name in SOURCE_ORDER:

    source_examples = [
        x
        for x in dev_examples
        if x["source"] == source_name
    ]

    balanced_examples.extend(
        source_examples[:25]
    )

assert len(
    balanced_examples
) == 100

roundtrip_rows = []

for name, tok in legal_tokenizers.items():

    failures = 0

    for ex in balanced_examples:

        original = ex["prompt"]

        ids = tok.encode(
            original,
            add_special_tokens=False
        )

        decoded = tok.decode(
            ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )

        if decoded != original:
            failures += 1

    roundtrip_rows.append({
        "tokenizer": name,
        "examples_tested": 100,
        "roundtrip_failures": failures,
    })

    print(
        f"{name:12s} | "
        f"failures={failures}"
    )

roundtrip_df = pd.DataFrame(
    roundtrip_rows
)

roundtrip_df.to_csv(
    RESULTS_DIR
    / "tokenizer_roundtrip_results.csv",
    index=False
)

assert all(
    roundtrip_df[
        "roundtrip_failures"
    ] == 0
)

efficiency_rows = []

for name, tok in {
    "GPT2_50K": audit_tokenizer,
    **legal_tokenizers,
}.items():

    total_tokens = 0
    total_words = 0

    for ex in dev_examples:

        text = ex["prompt"]

        total_words += len(
            text.split()
        )

        total_tokens += len(
            tok.encode(
                text,
                add_special_tokens=False
            )
        )

    efficiency_rows.append({
        "tokenizer": name,
        "vocab_size": len(tok),
        "total_tokens": total_tokens,
        "total_words": total_words,
        "tokens_per_word": (
            total_tokens / total_words
        ),
        "embedding_params_at_d1024": (
            len(tok) * 1024
        ),
    })

tokenizer_results = pd.DataFrame(
    efficiency_rows
)

gpt2_tpw = float(
    tokenizer_results.loc[
        tokenizer_results["tokenizer"]
        == "GPT2_50K",
        "tokens_per_word",
    ].iloc[0]
)

tokenizer_results[
    "reduction_vs_gpt2_pct"
] = (
    100.0
    * (
        1.0
        - tokenizer_results[
            "tokens_per_word"
        ] / gpt2_tpw
    )
)

tokenizer_results.to_csv(
    RESULTS_DIR
    / "tokenizer_efficiency_comparison.csv",
    index=False
)

display(tokenizer_results)

print(
    "Tokenizer round-trip: PASSED"
)


In [ ]:

FINAL_TOKENIZER_NAME = "LEGAL_40K"

FINAL_TOKENIZER_DIR = (
    LEGAL_40K_DIR
)

FINAL_TOKENIZER = (
    legal_tokenizer_40k
)

assert len(
    FINAL_TOKENIZER
) == 40000

with open(
    RESULTS_DIR
    / "final_tokenizer_selection.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "name": FINAL_TOKENIZER_NAME,
            "vocab_size": 40000,
            "selection": (
                "Current engineering trade-off "
                "from the 32K/40K/50K study."
            ),
        },
        f,
        indent=2
    )

print(
    "FINAL TOKENIZER:",
    FINAL_TOKENIZER_NAME
)
print(
    "VOCAB SIZE:",
    len(FINAL_TOKENIZER)
)


## Stage 2 — Architecture A (large serious-run model)


In [ ]:

MODEL_CONFIG = {
    "vocab_size": 40000,
    "context_length": 2048,

    "d_model": 1024,
    "n_layers": 18,

    "n_heads": 16,
    "n_kv_heads": 4,
    "head_dim": 64,

    "ffn_dim": 2816,

    "rope_theta": 10000.0,
    "norm_eps": 1e-5,

    "dropout": 0.0,
    "tie_embeddings": True,
    "use_bias": False,

    "gradient_checkpointing": True,
}

assert (
    MODEL_CONFIG["d_model"]
    == MODEL_CONFIG["n_heads"]
    * MODEL_CONFIG["head_dim"]
)

assert (
    MODEL_CONFIG["n_heads"]
    % MODEL_CONFIG["n_kv_heads"]
    == 0
)

print(
    json.dumps(
        MODEL_CONFIG,
        indent=2
    )
)


In [ ]:

def estimate_model_params(cfg):

    v = cfg["vocab_size"]
    d = cfg["d_model"]
    L = cfg["n_layers"]
    H = cfg["n_heads"]
    KV = cfg["n_kv_heads"]
    D = d // H
    FFD = cfg["ffn_dim"]

    embedding = v * d

    q = d * H * D
    k = d * KV * D
    val = d * KV * D
    o = H * D * d

    attention_per_layer = (
        q + k + val + o
    )

    ffn_per_layer = (
        3 * d * FFD
    )

    norm_params = (
        (2 * L + 1) * d
    )

    lm_head = (
        0
        if cfg["tie_embeddings"]
        else v * d
    )

    total = (
        embedding
        + L * (
            attention_per_layer
            + ffn_per_layer
        )
        + norm_params
        + lm_head
    )

    return {
        "embedding": embedding,
        "attention": (
            L * attention_per_layer
        ),
        "ffn": (
            L * ffn_per_layer
        ),
        "norm": norm_params,
        "lm_head": lm_head,
        "total": total,
    }


PARAM_BREAKDOWN = (
    estimate_model_params(
        MODEL_CONFIG
    )
)

TARGET_PARAMS = PARAM_BREAKDOWN[
    "total"
]

print("=" * 80)
print("ARCHITECTURE A PARAMETER BUDGET")
print("=" * 80)

for key, value in PARAM_BREAKDOWN.items():
    print(
        f"{key:15s}: "
        f"{value:,} "
        f"({value/1e6:.3f}M)"
    )

assert (
    50_000_000
    <= TARGET_PARAMS
    <= 300_000_000
)

print(
    f"\nTOTAL = {TARGET_PARAMS:,} "
    f"({TARGET_PARAMS/1e6:.3f}M)"
)


## Stage 3 — Robust token cache

The token cache is **document-transactional**. Progress is committed only after a complete document has been tokenized and its resulting partial buffer/shards have been safely written.

If a Kaggle session is interrupted halfway through a document, the next session reprocesses that document rather than silently losing its suffix.


In [ ]:

SHARD_TOKENS = 8_000_000
PROGRESS_EVERY_RECORDS = 250

EOS_ID = FINAL_TOKENIZER.eos_token_id

assert EOS_ID is not None
assert len(FINAL_TOKENIZER) < 65536

SOURCE_CACHE_DIRS = {
    source: TOKEN_CACHE_DIR / source
    for source in SOURCE_ORDER
}

for p in SOURCE_CACHE_DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Shard target:",
    f"{SHARD_TOKENS:,} tokens"
)

print(
    "Storage dtype: uint16"
)

print(
    "EOS token ID:",
    EOS_ID
)


In [ ]:

def atomic_json_save(obj, path):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2
        )

    os.replace(
        tmp,
        path
    )


def atomic_bin_write(
    array,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    np.asarray(
        array,
        dtype=np.uint16
    ).tofile(tmp)

    os.replace(
        tmp,
        path
    )


def atomic_npy_write(
    array,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        ".tmp.npy"
    )

    np.save(
        tmp,
        np.asarray(
            array,
            dtype=np.uint16
        )
    )

    os.replace(
        tmp,
        path
    )


def load_json(
    path,
    default=None
):

    path = Path(path)

    if not path.exists():
        return default

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


print(
    "Atomic helpers: READY"
)


In [ ]:

def load_source_state(
    source_dir,
    source_name
):

    state_path = (
        Path(source_dir)
        / "state.json"
    )

    if state_path.exists():

        with open(
            state_path,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    return {
        "source": source_name,
        "tokenizer": FINAL_TOKENIZER_NAME,
        "vocab_size": len(FINAL_TOKENIZER),
        "dtype": "uint16",
        "shard_tokens_target": SHARD_TOKENS,

        "next_record_index": 0,
        "next_shard_id": 0,
        "records_used": 0,
        "total_tokens": 0,

        "completed": False,
        "shards": [],
    }


def save_source_state(
    source_dir,
    state
):

    atomic_json_save(
        state,
        Path(source_dir)
        / "state.json"
    )


def clean_uncommitted_shards(
    source_dir,
    next_shard_id
):

    for p in Path(
        source_dir
    ).glob("shard_*.bin"):

        m = re.search(
            r"shard_(\d+)\.bin$",
            p.name
        )

        if (
            m
            and int(m.group(1))
            >= int(next_shard_id)
        ):

            try:
                p.unlink()
            except OSError:
                pass


def build_source_token_cache(
    source_name
):

    source_dir = SOURCE_CACHE_DIRS[
        source_name
    ]

    source_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    state = load_source_state(
        source_dir,
        source_name
    )

    # Never reuse a cache produced by a different tokenizer/model input.
    if state.get("tokenizer") != FINAL_TOKENIZER_NAME:
        raise RuntimeError(
            f"Existing token cache for {source_name} was built with "
            f"{state.get('tokenizer')!r}, not {FINAL_TOKENIZER_NAME!r}. "
            "Delete that source cache or use the matching tokenizer."
        )

    if int(state.get("vocab_size", -1)) != len(FINAL_TOKENIZER):
        raise RuntimeError(
            f"Existing token cache for {source_name} has vocab size "
            f"{state.get('vocab_size')}, expected {len(FINAL_TOKENIZER)}."
        )

    if state["completed"]:

        print(
            f"{source_name:20s} | "
            f"already complete | "
            f"tokens={state['total_tokens']:,}"
        )

        return state

    next_record_index = int(
        state["next_record_index"]
    )

    next_shard_id = int(
        state["next_shard_id"]
    )

    clean_uncommitted_shards(
        source_dir,
        next_shard_id
    )

    partial_path = (
        source_dir
        / "partial_buffer.npy"
    )

    if partial_path.exists():

        partial = np.load(
            partial_path,
            allow_pickle=False
        ).astype(
            np.uint16,
            copy=False
        )

    else:

        partial = np.empty(
            0,
            dtype=np.uint16
        )

    started = time.time()

    with open(
        downloaded_files[source_name],
        "rb"
    ) as f:

        for raw_index, record in enumerate(
            ijson.items(
                f,
                "item"
            )
        ):

            if raw_index < (
                next_record_index
            ):
                continue

            # Exact holdout exclusion
            if (
                source_name,
                raw_index
            ) in holdout_keys:
                continue

            if not isinstance(
                record,
                dict
            ):
                continue

            text = record.get(
                "text",
                ""
            )

            if (
                not isinstance(
                    text,
                    str
                )
                or not text.strip()
            ):
                continue

            # Tokenize one full document
            ids = FINAL_TOKENIZER.encode(
                text,
                add_special_tokens=False
            )

            if not ids:
                continue

            ids.append(
                EOS_ID
            )

            doc_tokens = np.asarray(
                ids,
                dtype=np.uint16
            )

            if len(partial):

                combined = np.concatenate(
                    [
                        partial,
                        doc_tokens
                    ]
                )

            else:

                combined = doc_tokens

            offset = 0
            new_shards = []

            while (
                len(combined) - offset
                >= SHARD_TOKENS
            ):

                shard_id = (
                    next_shard_id
                    + len(new_shards)
                )

                shard = combined[
                    offset:
                    offset + SHARD_TOKENS
                ]

                shard_path = (
                    source_dir
                    / f"shard_{shard_id:06d}.bin"
                )

                atomic_bin_write(
                    shard,
                    shard_path
                )

                new_shards.append({
                    "id": shard_id,
                    "file": str(
                        shard_path
                    ),
                    "tokens": SHARD_TOKENS,
                })

                offset += SHARD_TOKENS

            new_partial = combined[
                offset:
            ].copy()

            # Transaction commit at document
            # boundary.
            committed_shards = (
                list(state["shards"])
                + new_shards
            )

            committed_total = (
                sum(
                    int(x["tokens"])
                    for x in committed_shards
                )
                + len(new_partial)
            )

            if len(new_partial):

                atomic_npy_write(
                    new_partial,
                    partial_path
                )

            elif partial_path.exists():

                partial_path.unlink()

            candidate_state = {
                "source": source_name,
                "tokenizer": FINAL_TOKENIZER_NAME,
                "vocab_size": len(
                    FINAL_TOKENIZER
                ),
                "dtype": "uint16",
                "shard_tokens_target": SHARD_TOKENS,

                "next_record_index": (
                    raw_index + 1
                ),

                "next_shard_id": (
                    next_shard_id
                    + len(new_shards)
                ),

                "records_used": (
                    int(
                        state[
                            "records_used"
                        ]
                    )
                    + 1
                ),

                "total_tokens": int(
                    committed_total
                ),

                "completed": False,
                "shards": committed_shards,
            }

            # SINGLE atomic state commit.
            save_source_state(
                source_dir,
                candidate_state
            )

            state = candidate_state
            partial = new_partial

            next_shard_id = int(
                state["next_shard_id"]
            )

            if (
                state["records_used"]
                % PROGRESS_EVERY_RECORDS
                == 0
            ):

                hours = (
                    time.time()
                    - started
                ) / 3600.0

                print(
                    f"{source_name:20s} | "
                    f"raw={raw_index:,} | "
                    f"records={state['records_used']:,} | "
                    f"tokens={state['total_tokens']:,} | "
                    f"elapsed={hours:.2f}h"
                )

    # Commit final partial shard
    if len(partial):

        final_shard_path = (
            source_dir
            / f"shard_{next_shard_id:06d}.bin"
        )

        atomic_bin_write(
            partial,
            final_shard_path
        )

        state["shards"].append({
            "id": next_shard_id,
            "file": str(
                final_shard_path
            ),
            "tokens": int(
                len(partial)
            ),
        })

        next_shard_id += 1
        partial = np.empty(
            0,
            dtype=np.uint16
        )

        if partial_path.exists():
            partial_path.unlink()

    state["next_record_index"] = -1
    state["next_shard_id"] = (
        next_shard_id
    )
    state["total_tokens"] = sum(
        int(x["tokens"])
        for x in state["shards"]
    )
    state["completed"] = True

    save_source_state(
        source_dir,
        state
    )

    print(
        f"{source_name:20s} | COMPLETE | "
        f"tokens={state['total_tokens']:,} | "
        f"shards={len(state['shards']):,}"
    )

    return state


In [ ]:

source_manifests = {}

for source_name in SOURCE_ORDER:

    print(
        "\n"
        + "=" * 80
    )

    print(
        "TOKEN CACHE:",
        source_name.upper()
    )

    source_manifests[
        source_name
    ] = build_source_token_cache(
        source_name
    )

    print(
        "=" * 80
    )

cache_rows = []

for source_name in SOURCE_ORDER:

    m = source_manifests[
        source_name
    ]

    cache_rows.append({
        "source": source_name,
        "tokens": int(
            m["total_tokens"]
        ),
        "shards": len(
            m["shards"]
        ),
        "completed": bool(
            m["completed"]
        ),
    })

cache_summary_df = pd.DataFrame(
    cache_rows
)

display(cache_summary_df)

assert all(
    cache_summary_df[
        "completed"
    ]
)

TOTAL_TRAIN_TOKENS = int(
    cache_summary_df[
        "tokens"
    ].sum()
)

print(
    "\nTOTAL HOLDOUT-EXCLUDED TRAINING TOKENS:",
    f"{TOTAL_TRAIN_TOKENS:,}"
)

atomic_json_save(
    {
        "tokenizer": FINAL_TOKENIZER_NAME,
        "vocab_size": len(FINAL_TOKENIZER),
        "sources": source_manifests,
        "total_tokens": TOTAL_TRAIN_TOKENS,
    },
    TOKEN_CACHE_DIR
    / "global_manifest.json"
)


In [ ]:

global_manifest_path = (
    TOKEN_CACHE_DIR
    / "global_manifest.json"
)

with open(
    global_manifest_path,
    "r",
    encoding="utf-8"
) as f:

    global_manifest = json.load(f)

assert (
    global_manifest["tokenizer"]
    == FINAL_TOKENIZER_NAME
)

assert (
    global_manifest["vocab_size"]
    == len(FINAL_TOKENIZER)
)

for source_name in SOURCE_ORDER:

    m = global_manifest[
        "sources"
    ][source_name]

    running_tokens = 0

    expected_ids = list(
        range(
            len(m["shards"])
        )
    )

    actual_ids = [
        int(x["id"])
        for x in m["shards"]
    ]

    assert (
        actual_ids == expected_ids
    )

    for shard in m["shards"]:

        p = Path(
            shard["file"]
        )

        assert p.exists(), (
            f"Missing shard: {p}"
        )

        expected_bytes = (
            int(shard["tokens"])
            * 2
        )

        assert (
            p.stat().st_size
            == expected_bytes
        )

        running_tokens += int(
            shard["tokens"]
        )

    assert (
        running_tokens
        == int(
            m["total_tokens"]
        )
    )

print(
    "TOKEN CACHE INTEGRITY: PASSED"
)

print(
    "Total cached tokens:",
    f"{global_manifest['total_tokens']:,}"
)


In [ ]:

VALIDATION_BIN = (
    TRAIN_RESULTS_DIR
    / "validation.bin"
)

VALIDATION_META = (
    TRAIN_RESULTS_DIR
    / "validation_meta.json"
)

if not (
    VALIDATION_BIN.exists()
    and VALIDATION_META.exists()
):

    validation_ids = []

    for ex in dev_examples:

        full_text = (
            ex["prompt"]
            + " "
            + ex["reference"]
        )

        ids = FINAL_TOKENIZER.encode(
            full_text,
            add_special_tokens=False
        )

        ids.append(
            EOS_ID
        )

        validation_ids.extend(
            ids
        )

    validation_array = np.asarray(
        validation_ids,
        dtype=np.uint16
    )

    atomic_bin_write(
        validation_array,
        VALIDATION_BIN
    )

    atomic_json_save(
        {
            "tokens": int(
                len(validation_array)
            ),
            "dtype": "uint16",
            "examples": len(
                dev_examples
            ),
        },
        VALIDATION_META
    )

    del validation_array
    del validation_ids

    gc.collect()

with open(
    VALIDATION_META,
    "r",
    encoding="utf-8"
) as f:

    validation_meta = json.load(f)

assert (
    VALIDATION_BIN.stat().st_size
    == int(
        validation_meta[
            "tokens"
        ]
    ) * 2
)

print(
    "Validation tokens:",
    f"{validation_meta['tokens']:,}"
)

print(
    "Validation cache: READY"
)


## Stage 4 — Architecture A model


In [ ]:

class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-5
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        inv_rms = torch.rsqrt(
            x.pow(2).mean(
                dim=-1,
                keepdim=True
            )
            + self.eps
        )

        return (
            x
            * inv_rms
            * self.weight
        )


class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        dim,
        max_seq_len,
        theta=10000.0
    ):

        super().__init__()

        if dim % 2:
            raise ValueError(
                "RoPE dimension must be even."
            )

        inv_freq = 1.0 / (
            theta ** (
                torch.arange(
                    0,
                    dim,
                    2,
                    dtype=torch.float32
                )
                / dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32
        )

        freqs = torch.outer(
            positions,
            inv_freq
        )

        self.register_buffer(
            "cos_cached",
            torch.repeat_interleave(
                freqs.cos(),
                2,
                dim=-1
            ),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            torch.repeat_interleave(
                freqs.sin(),
                2,
                dim=-1
            ),
            persistent=False
        )

    def forward(
        self,
        x,
        position_offset=0
    ):

        seq_len = x.size(-2)

        end = (
            position_offset
            + seq_len
        )

        if (
            end
            > self.cos_cached.size(0)
        ):
            raise ValueError(
                "RoPE context exceeded."
            )

        cos = self.cos_cached[
            position_offset:end
        ].to(
            device=x.device,
            dtype=x.dtype
        )

        sin = self.sin_cached[
            position_offset:end
        ].to(
            device=x.device,
            dtype=x.dtype
        )

        cos = cos.view(
            1,
            1,
            seq_len,
            -1
        )

        sin = sin.view(
            1,
            1,
            seq_len,
            -1
        )

        even = x[..., 0::2]
        odd = x[..., 1::2]

        rotated = torch.stack(
            [
                -odd,
                even
            ],
            dim=-1
        ).flatten(-2)

        return (
            x * cos
            + rotated * sin
        )


class GQAAttention(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        n_heads = cfg[
            "n_heads"
        ]

        n_kv_heads = cfg[
            "n_kv_heads"
        ]

        head_dim = (
            d_model // n_heads
        )

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim

        self.group_size = (
            n_heads
            // n_kv_heads
        )

        self.q_proj = nn.Linear(
            d_model,
            n_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.o_proj = nn.Linear(
            d_model,
            d_model,
            bias=cfg["use_bias"]
        )

        self.rope = RotaryEmbedding(
            head_dim,
            cfg["context_length"],
            cfg["rope_theta"]
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False
    ):

        batch, seq_len, d_model = (
            x.shape
        )

        past_len = (
            0
            if past_kv is None
            else past_kv[0].size(2)
        )

        q = self.q_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        k = self.k_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_kv_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        v = self.v_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_kv_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        q = self.rope(
            q,
            past_len
        )

        k = self.rope(
            k,
            past_len
        )

        if past_kv is not None:

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2
            )

            v = torch.cat(
                [past_v, v],
                dim=2
            )

        if (
            self.n_kv_heads
            != self.n_heads
        ):

            k_attn = (
                k.repeat_interleave(
                    self.group_size,
                    dim=1
                )
            )

            v_attn = (
                v.repeat_interleave(
                    self.group_size,
                    dim=1
                )
            )

        else:

            k_attn = k
            v_attn = v

        if past_kv is None:

            y = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=True,
            )

        else:

            y = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=False,
            )

        y = y.transpose(
            1,
            2
        ).contiguous().view(
            batch,
            seq_len,
            d_model
        )

        output = self.o_proj(
            y
        )

        if use_cache:
            return output, (k, v)

        return output


class SwiGLU(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        ffn_dim = cfg[
            "ffn_dim"
        ]

        self.gate_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=cfg["use_bias"]
        )

        self.up_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=cfg["use_bias"]
        )

        self.down_proj = nn.Linear(
            ffn_dim,
            d_model,
            bias=cfg["use_bias"]
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            * self.up_proj(x)
        )


class DecoderBlock(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        self.norm1 = RMSNorm(
            d_model,
            cfg["norm_eps"]
        )

        self.attn = GQAAttention(
            cfg
        )

        self.norm2 = RMSNorm(
            d_model,
            cfg["norm_eps"]
        )

        self.ffn = SwiGLU(
            cfg
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False
    ):

        if use_cache:

            attn_out, present = (
                self.attn(
                    self.norm1(x),
                    past_kv=past_kv,
                    use_cache=True,
                )
            )

            x = x + attn_out

            x = x + self.ffn(
                self.norm2(x)
            )

            return x, present

        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


class LegalDecoderA(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        self.cfg = dict(cfg)

        self.token_embedding = nn.Embedding(
            cfg["vocab_size"],
            cfg["d_model"]
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(cfg)
            for _ in range(
                cfg["n_layers"]
            )
        ])

        self.final_norm = RMSNorm(
            cfg["d_model"],
            cfg["norm_eps"]
        )

        self.lm_head = nn.Linear(
            cfg["d_model"],
            cfg["vocab_size"],
            bias=False
        )

        if cfg["tie_embeddings"]:

            self.lm_head.weight = (
                self.token_embedding.weight
            )

        self.apply(
            self._init_weights
        )

        residual_scale = (
            1.0
            / math.sqrt(
                2.0
                * cfg["n_layers"]
            )
        )

        for block in self.blocks:

            nn.init.normal_(
                block.attn.o_proj.weight,
                mean=0.0,
                std=0.02
                * residual_scale,
            )

            nn.init.normal_(
                block.ffn.down_proj.weight,
                mean=0.0,
                std=0.02
                * residual_scale,
            )

    def _init_weights(
        self,
        module
    ):

        if isinstance(
            module,
            nn.Linear
        ):

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(
            module,
            RMSNorm
        ):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        labels=None,
        use_cache=False,
        past_key_values=None
    ):

        x = self.token_embedding(
            input_ids
        )

        if use_cache:

            if past_key_values is None:

                past_key_values = [
                    None
                    for _ in self.blocks
                ]

            presents = []

            for block, past in zip(
                self.blocks,
                past_key_values
            ):

                x, present = block(
                    x,
                    past_kv=past,
                    use_cache=True
                )

                presents.append(
                    present
                )

            x = self.final_norm(
                x
            )

            logits = self.lm_head(
                x
            )

            return {
                "logits": logits,
                "loss": None,
                "past_key_values": presents,
            }

        for block in self.blocks:

            if (
                self.training
                and self.cfg[
                    "gradient_checkpointing"
                ]
            ):

                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    use_reentrant=False
                )

            else:

                x = block(x)

        x = self.final_norm(
            x
        )

        logits = self.lm_head(
            x
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(
                    -1
                )
            )

        return {
            "logits": logits,
            "loss": loss,
        }


print(
    "Architecture A model classes: READY"
)


In [ ]:

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_A = LegalDecoderA(
    MODEL_CONFIG
).to(DEVICE)

actual_params = sum(
    p.numel()
    for p in model_A.parameters()
)

print(
    "Actual parameters:",
    f"{actual_params:,}",
    f"({actual_params/1e6:.3f}M)"
)

assert (
    actual_params
    == TARGET_PARAMS
)

assert (
    50_000_000
    <= actual_params
    <= 300_000_000
)

print(
    "Parameter check: PASSED"
)


In [ ]:

model_A.train()

smoke_batch = 1
smoke_seq = 128

smoke_ids = torch.randint(
    0,
    MODEL_CONFIG["vocab_size"],
    (
        smoke_batch,
        smoke_seq
    ),
    dtype=torch.long,
    device=DEVICE
)

smoke_labels = smoke_ids.roll(
    shifts=-1,
    dims=1
)

smoke_labels[
    :,
    -1
] = -100

smoke_optimizer = torch.optim.AdamW(
    model_A.parameters(),
    lr=1e-4,
    betas=(0.9, 0.95),
    weight_decay=0.0,
)

smoke_optimizer.zero_grad(
    set_to_none=True
)

with (
    torch.amp.autocast(
        "cuda",
        dtype=torch.float16
    )
    if DEVICE.type == "cuda"
    else nullcontext()
):

    smoke_out = model_A(
        input_ids=smoke_ids,
        labels=smoke_labels
    )

smoke_loss = smoke_out[
    "loss"
]

assert smoke_loss is not None
assert torch.isfinite(
    smoke_loss
)

smoke_loss.backward()

smoke_grad = (
    torch.nn.utils.clip_grad_norm_(
        model_A.parameters(),
        1.0
    )
)

smoke_optimizer.step()

print(
    "Logits:",
    tuple(
        smoke_out["logits"].shape
    )
)

print(
    "Loss:",
    float(
        smoke_loss.detach().cpu()
    )
)

print(
    "Grad norm:",
    float(smoke_grad)
)

assert (
    smoke_out["logits"].shape
    == (
        smoke_batch,
        smoke_seq,
        MODEL_CONFIG[
            "vocab_size"
        ]
    )
)

print(
    "Forward/backward smoke test: PASSED"
)

del smoke_optimizer
del smoke_ids
del smoke_labels
del smoke_out

gc.collect()

torch.cuda.empty_cache()


In [ ]:

model_A.eval()

prompt_ids = FINAL_TOKENIZER.encode(
    "The court held that the petition",
    add_special_tokens=False
)

prompt_ids = prompt_ids[
    :16
]

probe = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():

    first = model_A(
        input_ids=probe,
        use_cache=True
    )

    next_id = torch.argmax(
        first["logits"][
            :,
            -1,
            :
        ],
        dim=-1
    ).unsqueeze(1)

    second = model_A(
        input_ids=next_id,
        use_cache=True,
        past_key_values=first[
            "past_key_values"
        ]
    )

assert (
    second["logits"].shape
    == (
        1,
        1,
        MODEL_CONFIG[
            "vocab_size"
        ]
    )
)

assert (
    len(
        second[
            "past_key_values"
        ]
    )
    == MODEL_CONFIG[
        "n_layers"
    ]
)

print(
    "KV-cache smoke test: PASSED"
)

del first
del second
del next_id
del probe

gc.collect()

torch.cuda.empty_cache()

model_A.train()


## Stage 5 — Training engine

The next cells implement:
- deterministic source-aware shard streaming;
- gradient accumulation;
- mixed precision;
- cosine decay with warmup;
- periodic validation;
- transactional checkpointing;
- exact stream-state resume after Kaggle interruption.


In [ ]:

# Memory / compute
PER_GPU_MICRO_BATCH = 1
GRADIENT_ACCUMULATION_STEPS = 16

# Optimizer
LEARNING_RATE = 2e-4
MIN_LEARNING_RATE = 2e-5
BETAS = (0.9, 0.95)
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
WARMUP_STEPS = 2000

# Validation/checkpoints
CHECKPOINT_EVERY_STEPS = 250
VALIDATE_EVERY_STEPS = 250
VALIDATION_BATCHES = 16

# Kaggle full-notebook safety.
SESSION_RESERVE_MINUTES = 10

# Source interleaving
STREAM_SEED = SEED + 9000
STREAM_TEMPERATURE = 0.7

# Long training switch
RUN_LONG_TRAINING = True

TRAINING_PLAN = {
    "architecture": "A_large",
    "parameters": int(actual_params),
    "tokenizer": FINAL_TOKENIZER_NAME,
    "vocab_size": len(FINAL_TOKENIZER),
    "context_length": MODEL_CONFIG["context_length"],

    "per_gpu_micro_batch": PER_GPU_MICRO_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,

    "learning_rate": LEARNING_RATE,
    "min_learning_rate": MIN_LEARNING_RATE,
    "betas": BETAS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,

    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "validate_every_steps": VALIDATE_EVERY_STEPS,
    "validation_batches": VALIDATION_BATCHES,

    "safe_total_runtime_minutes": SAFE_TOTAL_RUNTIME_MINUTES,
    "session_reserve_minutes": SESSION_RESERVE_MINUTES,

    "stream_seed": STREAM_SEED,
    "stream_temperature": STREAM_TEMPERATURE,

    "run_long_training": RUN_LONG_TRAINING,
}

atomic_json_save(
    TRAINING_PLAN,
    TRAIN_RESULTS_DIR
    / "training_plan.json"
)

print(
    json.dumps(
        TRAINING_PLAN,
        indent=2
    )
)


In [ ]:

class ShardCursor:

    def __init__(
        self,
        source_name,
        manifest
    ):

        self.source_name = source_name
        self.shards = manifest[
            "shards"
        ]

        self.shard_index = 0
        self.offset = 0
        self._mmap = None

    @property
    def exhausted(self):

        return (
            self.shard_index
            >= len(self.shards)
        )

    def _open(self):

        if self.exhausted:

            self._mmap = None
            return

        shard = self.shards[
            self.shard_index
        ]

        self._mmap = np.memmap(
            shard["file"],
            dtype=np.uint16,
            mode="r",
            shape=(int(
                shard["tokens"]
            ),),
        )

    def remaining_tokens(self):

        if self.exhausted:
            return 0

        if self._mmap is None:

            current = (
                int(
                    self.shards[
                        self.shard_index
                    ]["tokens"]
                )
                - self.offset
            )

        else:

            current = (
                len(self._mmap)
                - self.offset
            )

        later = sum(
            int(x["tokens"])
            for x in self.shards[
                self.shard_index + 1:
            ]
        )

        return max(
            current,
            0
        ) + later

    def read(self, n):

        remaining = int(n)
        pieces = []

        while (
            remaining > 0
            and not self.exhausted
        ):

            if self._mmap is None:
                self._open()

            available = (
                len(self._mmap)
                - self.offset
            )

            if available <= 0:

                self.shard_index += 1
                self.offset = 0
                self._mmap = None
                continue

            take = min(
                available,
                remaining
            )

            pieces.append(
                np.asarray(
                    self._mmap[
                        self.offset:
                        self.offset
                        + take
                    ]
                )
            )

            self.offset += take
            remaining -= take

            if (
                self.offset
                >= len(self._mmap)
            ):

                self.shard_index += 1
                self.offset = 0
                self._mmap = None

        if not pieces:
            return None

        if len(pieces) == 1:
            return pieces[0]

        return np.concatenate(
            pieces
        )

    def state_dict(self):

        return {
            "source_name": self.source_name,
            "shard_index": int(
                self.shard_index
            ),
            "offset": int(
                self.offset
            ),
        }

    def load_state_dict(
        self,
        state
    ):

        self.shard_index = int(
            state[
                "shard_index"
            ]
        )

        self.offset = int(
            state[
                "offset"
            ]
        )

        self._mmap = None


class MixedShardStream:

    def __init__(
        self,
        global_manifest,
        seed=STREAM_SEED,
        temperature=0.7,
    ):

        self.rng = random.Random(
            seed
        )

        self.temperature = float(
            temperature
        )

        self.cursors = {
            source: ShardCursor(
                source,
                global_manifest[
                    "sources"
                ][source]
            )
            for source in SOURCE_ORDER
        }

        self.base_tokens = {
            source: int(
                global_manifest[
                    "sources"
                ][source][
                    "total_tokens"
                ]
            )
            for source in SOURCE_ORDER
        }

    def active_sources(self):

        return [
            source
            for source, cursor
            in self.cursors.items()
            if not cursor.exhausted
        ]

    def choose_source(self):

        active = self.active_sources()

        if not active:
            return None

        weights = []

        for source in active:

            remaining = max(
                self.cursors[
                    source
                ].remaining_tokens(),
                1
            )

            base = max(
                self.base_tokens[
                    source
                ],
                1
            )

            fraction = (
                remaining
                / base
            )

            weights.append(
                fraction
                ** self.temperature
            )

        return self.rng.choices(
            active,
            weights=weights,
            k=1,
        )[0]

    def next_block(
        self,
        context_length
    ):

        while True:

            source = (
                self.choose_source()
            )

            if source is None:
                return None

            arr = self.cursors[
                source
            ].read(
                context_length + 1
            )

            if (
                arr is None
                or len(arr)
                < context_length + 1
            ):

                # Discard any tiny final tail.
                if not self.active_sources():
                    return None

                continue

            x = torch.from_numpy(
                arr[:-1].astype(
                    np.int64
                )
            )

            y = torch.from_numpy(
                arr[1:].astype(
                    np.int64
                )
            )

            return x, y, source

    def state_dict(self):

        return {
            "rng_state": self.rng.getstate(),
            "cursors": {
                source: cursor.state_dict()
                for source, cursor
                in self.cursors.items()
            },
        }

    def load_state_dict(
        self,
        state
    ):

        self.rng.setstate(
            state[
                "rng_state"
            ]
        )

        for source, cursor_state in (
            state[
                "cursors"
            ].items()
        ):

            self.cursors[
                source
            ].load_state_dict(
                cursor_state
            )


def make_training_batch(
    stream,
    batch_size,
    context_length,
):

    xs = []
    ys = []

    for _ in range(
        batch_size
    ):

        block = stream.next_block(
            context_length
        )

        if block is None:
            return None

        x, y, _ = block

        xs.append(x)
        ys.append(y)

    return (
        torch.stack(xs, dim=0),
        torch.stack(ys, dim=0),
    )


print(
    "Resumable shard stream: READY"
)


In [ ]:

def unwrap_model(
    model
):

    return (
        model.module
        if isinstance(
            model,
            nn.DataParallel
        )
        else model
    )


def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    stream,
    step,
    tokens_seen,
    best_val_loss,
    path,
):

    raw_model = unwrap_model(
        model
    )

    payload = {
        "model_state_dict": (
            raw_model.state_dict()
        ),

        "optimizer_state_dict": (
            optimizer.state_dict()
        ),

        "scheduler_state_dict": (
            scheduler.state_dict()
        ),

        "scaler_state_dict": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),

        "stream_state": (
            stream.state_dict()
        ),

        "step": int(step),
        "tokens_seen": int(tokens_seen),
        "best_val_loss": float(
            best_val_loss
        ),

        "model_config": MODEL_CONFIG,
        "training_plan": TRAINING_PLAN,
        "tokenizer_name": FINAL_TOKENIZER_NAME,
        "seed": SEED,

        "python_random_state": (
            random.getstate()
        ),

        "numpy_random_state": (
            np.random.get_state()
        ),

        "torch_random_state": (
            torch.get_rng_state()
        ),

        "cuda_random_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }

    path = Path(path)

    tmp = path.with_suffix(
        ".tmp.pt"
    )

    torch.save(
        payload,
        tmp
    )

    os.replace(
        tmp,
        path
    )


def save_best_model(
    model,
    val_loss
):

    raw_model = unwrap_model(
        model
    )

    path = (
        CHECKPOINT_DIR
        / "best_model.pt"
    )

    payload = {
        "model_state_dict": (
            raw_model.state_dict()
        ),

        "model_config": MODEL_CONFIG,
        "parameter_count": int(
            actual_params
        ),
        "tokenizer_name": (
            FINAL_TOKENIZER_NAME
        ),
        "validation_loss": float(
            val_loss
        ),
    }

    tmp = path.with_suffix(
        ".tmp.pt"
    )

    torch.save(
        payload,
        tmp
    )

    os.replace(
        tmp,
        path
    )


def load_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    stream,
):

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False,
    )

    assert (
        checkpoint[
            "model_config"
        ]
        == MODEL_CONFIG
    )

    assert (
        checkpoint[
            "tokenizer_name"
        ]
        == FINAL_TOKENIZER_NAME
    )

    unwrap_model(
        model
    ).load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    scheduler.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )

    if (
        scaler is not None
        and checkpoint[
            "scaler_state_dict"
        ] is not None
    ):

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

    stream.load_state_dict(
        checkpoint[
            "stream_state"
        ]
    )

    random.setstate(
        checkpoint[
            "python_random_state"
        ]
    )

    np.random.set_state(
        checkpoint[
            "numpy_random_state"
        ]
    )

    torch.set_rng_state(
        checkpoint[
            "torch_random_state"
        ]
    )

    saved_cuda_states = (
        checkpoint[
            "cuda_random_state"
        ]
    )

    if (
        torch.cuda.is_available()
        and saved_cuda_states is not None
        and len(saved_cuda_states)
        == torch.cuda.device_count()
    ):

        torch.cuda.set_rng_state_all(
            saved_cuda_states
        )

    return checkpoint


print(
    "Checkpoint utilities: READY"
)


In [ ]:

class WarmupCosineScheduler(
    torch.optim.lr_scheduler.LambdaLR
):

    def __init__(
        self,
        optimizer,
        warmup_steps,
        total_steps,
        min_ratio,
    ):

        warmup_steps = max(
            1,
            int(warmup_steps)
        )

        total_steps = max(
            warmup_steps + 1,
            int(total_steps)
        )

        min_ratio = float(
            min_ratio
        )

        def lr_lambda(step):

            if step < warmup_steps:

                return (
                    step
                    / warmup_steps
                )

            progress = (
                step
                - warmup_steps
            ) / (
                total_steps
                - warmup_steps
            )

            progress = min(
                1.0,
                max(
                    0.0,
                    progress
                )
            )

            cosine = 0.5 * (
                1.0
                + math.cos(
                    math.pi
                    * progress
                )
            )

            return (
                min_ratio
                + (
                    1.0
                    - min_ratio
                )
                * cosine
            )

        super().__init__(
            optimizer,
            lr_lambda
        )


def amp_context():

    if DEVICE.type == "cuda":

        return torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        )

    return nullcontext()


def make_scaler():

    if DEVICE.type != "cuda":
        return None

    try:

        return torch.amp.GradScaler(
            "cuda"
        )

    except TypeError:

        return torch.cuda.amp.GradScaler()


def evaluate_validation(
    model,
    max_batches=VALIDATION_BATCHES,
):

    raw = unwrap_model(
        model
    )

    was_training = raw.training

    raw.eval()

    n = int(
        validation_meta[
            "tokens"
        ]
    )

    mmap = np.memmap(
        VALIDATION_BIN,
        dtype=np.uint16,
        mode="r",
        shape=(n,),
    )

    max_start = max(
        0,
        n
        - MODEL_CONFIG[
            "context_length"
        ]
        - 1
    )

    starts = np.linspace(
        0,
        max_start,
        num=max_batches,
        dtype=np.int64,
    )

    losses = []

    with torch.no_grad():

        for start in starts:

            start = int(start)

            arr = np.asarray(
                mmap[
                    start:
                    start
                    + MODEL_CONFIG[
                        "context_length"
                    ]
                    + 1
                ]
            )

            if len(arr) != (
                MODEL_CONFIG[
                    "context_length"
                ]
                + 1
            ):
                continue

            x = torch.from_numpy(
                arr[:-1].astype(
                    np.int64
                )
            ).unsqueeze(
                0
            ).to(
                DEVICE,
                non_blocking=True
            )

            y = torch.from_numpy(
                arr[1:].astype(
                    np.int64
                )
            ).unsqueeze(
                0
            ).to(
                DEVICE,
                non_blocking=True
            )

            with amp_context():

                out = model(
                    input_ids=x,
                    labels=y
                )

            losses.append(
                float(
                    out["loss"]
                    .detach()
                    .cpu()
                )
            )

    del mmap

    if was_training:
        raw.train()

    if not losses:
        return float("inf")

    return float(
        np.mean(losses)
    )


print(
    "Validation and scheduler helpers: READY"
)


In [ ]:

if torch.cuda.device_count() >= 2:

    model_A = nn.DataParallel(
        model_A
    )

    print(
        "Training with DataParallel on",
        torch.cuda.device_count(),
        "GPUs."
    )

else:

    print(
        "Training on one GPU."
    )

global_micro_batch = (
    PER_GPU_MICRO_BATCH
    * max(
        torch.cuda.device_count(),
        1
    )
)

effective_tokens_per_update = (
    global_micro_batch
    * MODEL_CONFIG[
        "context_length"
    ]
    * GRADIENT_ACCUMULATION_STEPS
)

total_training_tokens = int(
    global_manifest[
        "total_tokens"
    ]
)

estimated_total_steps = math.ceil(
    total_training_tokens
    / effective_tokens_per_update
)

optimizer = torch.optim.AdamW(
    model_A.parameters(),
    lr=LEARNING_RATE,
    betas=BETAS,
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_steps=WARMUP_STEPS,
    total_steps=estimated_total_steps,
    min_ratio=(
        MIN_LEARNING_RATE
        / LEARNING_RATE
    )
)

scaler = make_scaler()

stream = MixedShardStream(
    global_manifest,
    seed=STREAM_SEED,
    temperature=STREAM_TEMPERATURE,
)

LATEST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "latest.pt"
)

BEST_MODEL_PATH = (
    CHECKPOINT_DIR
    / "best_model.pt"
)

start_step = 0
tokens_seen = 0
best_val_loss = float("inf")

if LATEST_CHECKPOINT.exists():

    checkpoint = load_checkpoint(
        LATEST_CHECKPOINT,
        model_A,
        optimizer,
        scheduler,
        scaler,
        stream,
    )

    start_step = int(
        checkpoint["step"]
    )

    tokens_seen = int(
        checkpoint["tokens_seen"]
    )

    best_val_loss = float(
        checkpoint["best_val_loss"]
    )

    print(
        "Resumed from checkpoint."
    )

    print(
        "Step:",
        start_step
    )

    print(
        "Tokens:",
        f"{tokens_seen:,}"
    )

else:

    print(
        "No checkpoint found — starting from scratch."
    )

print("=" * 80)
print("TRAINING ENGINE READY")
print("=" * 80)

print(
    "Effective tokens/update:",
    f"{effective_tokens_per_update:,}"
)

print(
    "Total training tokens:",
    f"{total_training_tokens:,}"
)

print(
    "Estimated optimizer steps:",
    f"{estimated_total_steps:,}"
)

print(
    "Current LR:",
    optimizer.param_groups[0]["lr"]
)


In [ ]:

test_stream = MixedShardStream(
    global_manifest,
    seed=STREAM_SEED + 1,
    temperature=STREAM_TEMPERATURE,
)

test_batch = make_training_batch(
    test_stream,
    batch_size=global_micro_batch,
    context_length=MODEL_CONFIG[
        "context_length"
    ],
)

assert test_batch is not None

tx, ty = test_batch

print(
    "x:",
    tuple(tx.shape)
)

print(
    "y:",
    tuple(ty.shape)
)

model_A.eval()

with torch.no_grad():

    tx_gpu = tx.to(
        DEVICE,
        non_blocking=True
    )

    out = model_A(
        input_ids=tx_gpu
    )

assert (
    out["logits"].shape
    == (
        global_micro_batch,
        MODEL_CONFIG[
            "context_length"
        ],
        MODEL_CONFIG[
            "vocab_size"
        ],
    )
)

print(
    "Stream/model integration: PASSED"
)

del test_stream
del test_batch
del tx
del ty
del tx_gpu
del out

gc.collect()

torch.cuda.empty_cache()

unwrap_model(
    model_A
).train()


## Stage 6 — Full-corpus resumable training

**This is the long-running cell.**

Run it on Kaggle. When the session reaches the safety limit, the notebook writes a checkpoint and stops. On the next Kaggle session, Run All again: the token cache is reused and `latest.pt` restores the model, optimizer, scheduler, RNG states, and shard cursor.


In [ ]:

if not RUN_LONG_TRAINING:

    print(
        "RUN_LONG_TRAINING=False — "
        "training skipped."
    )

else:

    session_start = time.time()

    elapsed_before_training = (
        session_start
        - NOTEBOOK_START_TIME
    )

    safe_runtime_seconds = (
        SAFE_TOTAL_RUNTIME_MINUTES
        * 60
    )

    remaining_runtime_seconds = (
        safe_runtime_seconds
        - elapsed_before_training
        - SESSION_RESERVE_MINUTES * 60
    )

    print(
        f"Safe training runtime remaining: "
        f"{max(0, remaining_runtime_seconds)/3600:.2f} h"
    )

    stop_reason = (
        "corpus_exhausted"
    )

    recent_losses = []

    optimizer.zero_grad(
        set_to_none=True
    )

    while True:

        if (
            time.time()
            - session_start
            >= max(0, remaining_runtime_seconds)
        ):

            stop_reason = (
                "session_time_limit"
            )

            break

        if not stream.active_sources():

            stop_reason = (
                "corpus_exhausted"
            )

            break

        optimizer.zero_grad(
            set_to_none=True
        )

        micro_losses = []
        microsteps = 0

        for _ in range(
            GRADIENT_ACCUMULATION_STEPS
        ):

            if (
                time.time()
                - session_start
                >= max(0, remaining_runtime_seconds)
            ):

                stop_reason = (
                    "session_time_limit"
                )

                break

            batch = make_training_batch(
                stream,
                batch_size=global_micro_batch,
                context_length=MODEL_CONFIG[
                    "context_length"
                ],
            )

            if batch is None:

                stop_reason = (
                    "corpus_exhausted"
                )

                break

            input_ids, labels = batch

            input_ids = input_ids.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            with amp_context():

                out = model_A(
                    input_ids=input_ids,
                    labels=labels,
                )

                loss = out[
                    "loss"
                ]

            # Accumulate raw gradients; after the microstep loop
            # we divide by the actual number of microsteps completed.
            if scaler is not None:

                scaler.scale(
                    loss
                ).backward()

            else:

                loss.backward()

            micro_losses.append(
                float(
                    loss.detach()
                    .cpu()
                )
            )

            microsteps += 1

            tokens_seen += (
                input_ids.numel()
            )

            del input_ids
            del labels
            del out

        if microsteps == 0:
            break

        if scaler is not None:

            scaler.unscale_(
                optimizer
            )

        # Average the accumulated gradient. This remains correct
        # if the Kaggle session timer interrupts an accumulation
        # window before all configured microsteps are completed.
        inv_microsteps = 1.0 / microsteps

        for parameter in model_A.parameters():

            if parameter.grad is not None:
                parameter.grad.mul_(
                    inv_microsteps
                )

        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                model_A.parameters(),
                MAX_GRAD_NORM
            )
        )

        if scaler is not None:

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            optimizer.step()

        scheduler.step()

        start_step += 1

        step_loss = float(
            np.mean(
                micro_losses
            )
        )

        recent_losses.append(
            step_loss
        )

        if len(recent_losses) > 100:

            recent_losses = (
                recent_losses[-100:]
            )

        if (
            start_step == 1
            or start_step % 25 == 0
        ):

            avg25 = float(
                np.mean(
                    recent_losses[-25:]
                )
            )

            elapsed_hours = (
                time.time()
                - session_start
            ) / 3600.0

            print(
                f"step={start_step:,} | "
                f"tokens={tokens_seen:,} | "
                f"loss={step_loss:.4f} | "
                f"avg25={avg25:.4f} | "
                f"grad={float(grad_norm):.3f} | "
                f"lr={optimizer.param_groups[0]['lr']:.6g} | "
                f"time={elapsed_hours:.2f}h"
            )

        if (
            start_step
            % VALIDATE_EVERY_STEPS
            == 0
        ):

            val_loss = evaluate_validation(
                model_A
            )

            val_ppl = math.exp(
                min(
                    val_loss,
                    20.0
                )
            )

            print(
                "=" * 80
            )

            print(
                f"VALIDATION | "
                f"step={start_step:,} | "
                f"loss={val_loss:.5f} | "
                f"ppl={val_ppl:.3f}"
            )

            print(
                "=" * 80
            )

            if (
                val_loss
                < best_val_loss
            ):

                best_val_loss = (
                    val_loss
                )

                save_best_model(
                    model_A,
                    best_val_loss
                )

                print(
                    "New best_model.pt saved."
                )

        if (
            start_step
            % CHECKPOINT_EVERY_STEPS
            == 0
        ):

            save_checkpoint(
                model_A,
                optimizer,
                scheduler,
                scaler,
                stream,
                start_step,
                tokens_seen,
                best_val_loss,
                LATEST_CHECKPOINT,
            )

            print(
                "Checkpoint saved:",
                LATEST_CHECKPOINT
            )

            gc.collect()

            torch.cuda.empty_cache()

    # Final validation at session exit.
    final_val_loss = (
        evaluate_validation(
            model_A
        )
    )

    if (
        not math.isfinite(
            best_val_loss
        )
        or final_val_loss
        < best_val_loss
    ):

        best_val_loss = (
            final_val_loss
        )

        save_best_model(
            model_A,
            best_val_loss
        )

        print(
            "Final validation model saved as best."
        )

    save_checkpoint(
        model_A,
        optimizer,
        scheduler,
        scaler,
        stream,
        start_step,
        tokens_seen,
        best_val_loss,
        LATEST_CHECKPOINT,
    )

    state = {
        "step": int(
            start_step
        ),
        "tokens_seen": int(
            tokens_seen
        ),
        "best_validation_loss": float(
            best_val_loss
        ),
        "stop_reason": stop_reason,
        "session_hours": (
            (
                time.time()
                - session_start
            )
            / 3600.0
        ),
        "total_training_tokens": int(
            total_training_tokens
        ),
        "progress_pct": (
            100.0
            * tokens_seen
            / max(
                total_training_tokens,
                1
            )
        ),
        "timestamp": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    atomic_json_save(
        state,
        TRAIN_RESULTS_DIR
        / "latest_training_state.json"
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        "TRAINING SESSION FINISHED"
    )

    print(
        json.dumps(
            state,
            indent=2
        )
    )

    print(
        "=" * 80
    )


## Stage 7 — Official test inference and submission

The final generation cell is deliberately separate from intermediate training sessions.

- During partial training, submission generation is deferred.
- After the complete holdout-excluded corpus pass, it can run automatically.
- An explicit override lets you create an interim submission from the current best checkpoint when you deliberately choose to do so.


In [ ]:

FORCE_SUBMISSION_GENERATION = False

AUTO_GENERATE_AFTER_FULL_CORPUS = False

GENERATION_MAX_NEW_TOKENS = 512
GENERATION_TEMPERATURE = 0.15
GENERATION_TOP_P = 0.95
GENERATION_TOP_K = 50

print(
    "Force submission:",
    FORCE_SUBMISSION_GENERATION
)

print(
    "Auto after full corpus:",
    AUTO_GENERATE_AFTER_FULL_CORPUS
)

print(
    "Temperature:",
    GENERATION_TEMPERATURE
)

print(
    "Top-p:",
    GENERATION_TOP_P
)

print(
    "Top-k:",
    GENERATION_TOP_K
)


In [ ]:

SPECIAL_TOKEN_IDS = {
    x
    for x in [
        FINAL_TOKENIZER.pad_token_id,
        FINAL_TOKENIZER.bos_token_id,
        FINAL_TOKENIZER.eos_token_id,
        FINAL_TOKENIZER.unk_token_id,
    ]
    if x is not None
}


def filter_logits(
    logits,
    top_k=0,
    top_p=1.0
):

    if (
        top_k
        and top_k > 0
    ):

        k = min(
            top_k,
            logits.size(-1)
        )

        threshold = torch.topk(
            logits,
            k
        ).values[
            ...,
            -1,
            None
        ]

        logits = torch.where(
            logits < threshold,
            torch.full_like(
                logits,
                float("-inf")
            ),
            logits
        )

    if top_p < 1.0:

        sorted_logits, sorted_idx = (
            torch.sort(
                logits,
                descending=True
            )
        )

        probs = torch.softmax(
            sorted_logits,
            dim=-1
        )

        cumulative = torch.cumsum(
            probs,
            dim=-1
        )

        remove = (
            cumulative
            - probs
            > top_p
        )

        sorted_logits = (
            sorted_logits.masked_fill(
                remove,
                float("-inf")
            )
        )

        logits = torch.full_like(
            logits,
            float("-inf")
        ).scatter(
            -1,
            sorted_idx,
            sorted_logits
        )

    return logits


@torch.no_grad()
def generate_exact_128_words(
    model,
    prompt_text,
    temperature=0.15,
    top_p=0.95,
    top_k=50,
    max_new_tokens=512,
):

    prompt_ids = FINAL_TOKENIZER.encode(
        prompt_text,
        add_special_tokens=False
    )

    # Preserve as much of the 512-word prompt as the context permits.
    max_prompt_tokens = (
        MODEL_CONFIG[
            "context_length"
        ]
        - 1
    )

    if len(prompt_ids) > (
        max_prompt_tokens
    ):

        prompt_ids = prompt_ids[
            -max_prompt_tokens:
        ]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    with amp_context():

        out = model(
            input_ids=input_ids,
            use_cache=True,
        )

    past = out[
        "past_key_values"
    ]

    logits = out[
        "logits"
    ][:, -1, :]

    generated = []

    for _ in range(
        max_new_tokens
    ):

        logits_step = logits.float()

        for token_id in (
            SPECIAL_TOKEN_IDS
        ):

            logits_step[
                :,
                token_id
            ] = float("-inf")

        if temperature <= 0:

            next_token = torch.argmax(
                logits_step,
                dim=-1
            )

        else:

            logits_step = (
                logits_step
                / temperature
            )

            logits_step = filter_logits(
                logits_step,
                top_k=top_k,
                top_p=top_p,
            )

            probs = torch.softmax(
                logits_step,
                dim=-1
            )

            next_token = (
                torch.multinomial(
                    probs,
                    num_samples=1
                ).squeeze(-1)
            )

        generated.append(
            int(
                next_token.item()
            )
        )

        decoded = FINAL_TOKENIZER.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        words = decoded.split()

        if len(words) >= 128:

            return " ".join(
                words[:128]
            )

        with amp_context():

            out = model(
                input_ids=next_token.view(
                    1,
                    1
                ),
                use_cache=True,
                past_key_values=past,
            )

        past = out[
            "past_key_values"
        ]

        logits = out[
            "logits"
        ][:, -1, :]

    decoded = FINAL_TOKENIZER.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    words = decoded.split()

    raise RuntimeError(
        "Model failed to produce 128 "
        f"whitespace-separated words "
        f"within {max_new_tokens} new tokens. "
        f"Got {len(words)} words."
    )


print(
    "Generation helpers: READY"
)


In [ ]:

prediction_path = (
    SUBMISSION_DIR
    / "prediction.json"
)

submission_zip_path = (
    SUBMISSION_DIR
    / "submission.zip"
)

training_state_path = (
    TRAIN_RESULTS_DIR
    / "latest_training_state.json"
)

best_available = (
    BEST_MODEL_PATH.exists()
)

training_finished = False

if training_state_path.exists():

    with open(
        training_state_path,
        "r",
        encoding="utf-8"
    ) as f:

        latest_training_state = (
            json.load(f)
        )

    training_finished = (
        latest_training_state.get(
            "stop_reason"
        )
        == "corpus_exhausted"
    )

should_generate = (
    best_available
    and (
        FORCE_SUBMISSION_GENERATION
        or (
            AUTO_GENERATE_AFTER_FULL_CORPUS
            and training_finished
        )
    )
)

print(
    "Best checkpoint available:",
    best_available
)

print(
    "Full corpus training finished:",
    training_finished
)

print(
    "Generate submission now:",
    should_generate
)

if not best_available:

    print(
        "No best_model.pt yet. "
        "Submission generation is deferred."
    )

elif not should_generate:

    print(
        "Submission generation deferred "
        "until full-corpus training finishes "
        "or FORCE_SUBMISSION_GENERATION=True."
    )

else:

    best_payload = torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    assert (
        best_payload[
            "model_config"
        ]
        == MODEL_CONFIG
    )

    assert (
        best_payload[
            "tokenizer_name"
        ]
        == FINAL_TOKENIZER_NAME
    )

    inference_model = LegalDecoderA(
        MODEL_CONFIG
    ).to(DEVICE)

    inference_model.load_state_dict(
        best_payload[
            "model_state_dict"
        ]
    )

    inference_model.eval()

    print(
        "Best model loaded for inference."
    )

    print(
        "Validation loss:",
        best_payload[
            "validation_loss"
        ]
    )


In [ ]:

if (
    best_available
    and should_generate
):

    predictions = {}

    started = time.time()

    for i, record in enumerate(
        test_records
    ):

        continuation = (
            generate_exact_128_words(
                inference_model,
                record["text"],
                temperature=(
                    GENERATION_TEMPERATURE
                ),
                top_p=GENERATION_TOP_P,
                top_k=GENERATION_TOP_K,
                max_new_tokens=(
                    GENERATION_MAX_NEW_TOKENS
                ),
            )
        )

        count = len(
            continuation.split()
        )

        assert count == 128

        predictions[
            record["id"]
        ] = {
            "text": continuation
        }

        if (
            (i + 1) % 10 == 0
            or i + 1
            == len(test_records)
        ):

            elapsed = (
                time.time()
                - started
            ) / 60.0

            print(
                f"Generated "
                f"{i+1:,}/"
                f"{len(test_records):,} "
                f"| elapsed={elapsed:.1f} min"
            )

    official_ids = {
        r["id"]
        for r in test_records
    }

    assert (
        set(predictions)
        == official_ids
    )

    assert (
        len(predictions)
        == 1000
    )

    assert all(
        len(
            item["text"].split()
        ) == 128
        for item in predictions.values()
    )

    with open(
        prediction_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2
        )

    with zipfile.ZipFile(
        submission_zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as zf:

        zf.write(
            prediction_path,
            arcname="prediction.json"
        )

    # Verify archive structure exactly.
    with zipfile.ZipFile(
        submission_zip_path,
        "r"
    ) as zf:

        names = zf.namelist()

        assert names == [
            "prediction.json"
        ]

        with zf.open(
            "prediction.json"
        ) as member:

            packaged = json.load(
                member
            )

    assert packaged == predictions

    print("=" * 80)
    print("SUBMISSION READY")
    print("=" * 80)

    print(
        "prediction.json:",
        prediction_path
    )

    print(
        "submission.zip:",
        submission_zip_path
    )

    print(
        "Predictions:",
        len(predictions)
    )

else:

    print(
        "No submission generated in this session."
    )


In [ ]:

print("=" * 80)
print("CS-781 PHASE 1 PIPELINE STATUS")
print("=" * 80)

status = {
    "official_train_sources": (
        len(downloaded_files) == 4
    ),

    "official_test_examples": (
        len(test_records) == 1000
    ),

    "official_test_prompts_512": all(
        len(r["text"].split()) == 512
        for r in test_records
    ),

    "holdout_examples": (
        len(holdout_keys) == 1000
    ),

    "dev_examples": (
        len(dev_examples) == 1000
    ),

    "dev_prompt_512": all(
        len(x["prompt"].split()) == 512
        for x in dev_examples
    ),

    "dev_reference_128": all(
        len(x["reference"].split()) == 128
        for x in dev_examples
    ),

    "legal_tokenizer_40k": (
        len(FINAL_TOKENIZER) == 40000
    ),

    "architecture_A_param_range": (
        50_000_000
        <= actual_params
        <= 300_000_000
    ),

    "token_cache_complete": all(
        global_manifest[
            "sources"
        ][source]["completed"]
        for source in SOURCE_ORDER
    ),

    "latest_checkpoint_exists": (
        LATEST_CHECKPOINT.exists()
    ),

    "best_model_exists": (
        BEST_MODEL_PATH.exists()
    ),

    "prediction_json_exists": (
        prediction_path.exists()
    ),

    "submission_zip_exists": (
        submission_zip_path.exists()
    ),
}

for name, passed in status.items():

    print(
        f"{'PASS' if passed else 'WAIT':4s} | {name}"
    )

print()
print(
    "WAIT is expected for artifacts that only "
    "appear after later training stages."
)

print(
    "\nNEXT REQUIRED DEVELOPMENT STAGE: "
    "Architecture B/C/D and controlled comparison."
)


## Architecture exploration requirement

This notebook completes the stable Architecture-A pipeline. It does **not** falsely claim that A/B/C/D are the same architecture with different hyperparameters.

The official capstone requires at least four substantially different architectures. After A is trained and validated, we will reuse the same corpus/tokenizer/holdout infrastructure for the other three architecture families and then compare the four models before selecting the final Phase-1 model set.
